In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_crime_data.csv", parse_dates=["Date Rptd", "DATE OCC"])
print("Input shape:", df.shape)
df.head()

Input shape: (743328, 24)


,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,LOCATION,LAT,LON
0,10304468,2020-01-08,2020-01-08,2230,3,Southwest,377,2,624,BATTERY - SIMPLE ASSAULT,...,501.0,SINGLE FAMILY DWELLING,400.0,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",AO,Adult Other,624.0,1100 W 39TH PL,34.0141,-118.2978
1,190101086,2020-01-02,2020-01-01,330,1,Central,163,2,624,BATTERY - SIMPLE ASSAULT,...,102.0,SIDEWALK,500.0,UNKNOWN WEAPON/OTHER WEAPON,IC,Invest Cont,624.0,700 S HILL ST,34.0459,-118.2545
2,200110444,2020-04-14,2020-02-13,1200,1,Central,155,2,845,SEX OFFENDER REGISTRANT OUT OF COMPLIANCE,...,726.0,POLICE FACILITY,0.0,No Weapon,AA,Adult Arrest,845.0,200 E 6TH ST,34.0448,-118.2474
3,191501505,2020-01-01,2020-01-01,1730,15,N Hollywood,1543,2,745,VANDALISM - MISDEAMEANOR ($399 OR UNDER),...,502.0,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",0.0,No Weapon,IC,Invest Cont,745.0,5400 CORTEEN PL,34.1685,-118.4019
4,191921269,2020-01-01,2020-01-01,415,19,Mission,1998,2,740,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...",...,409.0,BEAUTY SUPPLY STORE,0.0,No Weapon,IC,Invest Cont,740.0,14400 TITUS ST,34.2198,-118.4468


In [2]:
# Time-based features
df["Crime Hour"] = (df["TIME OCC"] // 100).astype(int).clip(0, 23)
df["Day of Week"] = df["DATE OCC"].dt.day_name()
df["Month"] = df["DATE OCC"].dt.month
df["Month Name"] = df["DATE OCC"].dt.month_name()
df["Year"] = df["DATE OCC"].dt.year

df[["Crime Hour", "Day of Week", "Month", "Month Name", "Year"]].head()

,Crime Hour,Day of Week,Month,Month Name,Year
0,22,Wednesday,1,January,2020
1,3,Wednesday,1,January,2020
2,12,Thursday,2,February,2020
3,17,Wednesday,1,January,2020
4,4,Wednesday,1,January,2020


In [3]:
# Weapon used flag
df["Weapon Used Flag"] = (df["Weapon Used Cd"] != 0).astype(int)
df["Weapon Used Flag"].value_counts()

Weapon Used Flag
0    485287
1    258041
Name: count, dtype: int64

In [4]:
# Violent crime flag - based on LAPD Part 1 violent crime categories
VIOLENT_KEYWORDS = [
    "HOMICIDE", "MANSLAUGHTER", "RAPE", "ROBBERY", "ASSAULT",
    "BATTERY", "KIDNAPPING", "SHOTS FIRED", "CRIMINAL THREATS",
    "LYNCHING", "STALKING",
]
pattern = "|".join(VIOLENT_KEYWORDS)
df["Violent Crime Flag"] = df["Crm Cd Desc"].str.contains(pattern, case=False, na=False).astype(int)

print(f"Violent crime share: {df['Violent Crime Flag'].mean():.2%}")
df["Violent Crime Flag"].value_counts()

Violent crime share: 29.18%


Violent Crime Flag
0    526423
1    216905
Name: count, dtype: int64

In [5]:
# Area-level crime volume
area_counts = df["AREA NAME"].value_counts()
df["Area Crime Count"] = df["AREA NAME"].map(area_counts)
df[["AREA NAME", "Area Crime Count"]].drop_duplicates().sort_values("Area Crime Count", ascending=False)

,AREA NAME,Area Crime Count
1,Central,49843
48,77th Street,47067
45,Pacific,43340
0,Southwest,41567
105,Hollywood,39696
98,Southeast,38083
313,Olympic,37741
3,N Hollywood,37011
109,Newton,36858
36,Wilshire,35102


In [6]:
# Save feature-engineered dataset
df.to_csv("../data/processed/feature_engineered_crime_data.csv", index=False)
print("Saved. Final shape:", df.shape)

Saved. Final shape: (743328, 32)
